<a href="https://colab.research.google.com/github/wbc-mjlab/wbc-mjlab/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# wbc-mjlab demo (G1 whole-body tracking)

Interactive **Viser** play with the bundled checkpoint and **samples** clip library (13 motions).

Steps: `pip install wbc-mjlab` → clone repo (samples + checkpoint) → convert CSVs to NPZ → visualize → `wbc-mjlab-demo`.

> **Runtime:** GPU (T4+). First run ~10–15 min (dependency sync + FK conversion).

## 1. GPU + headless MuJoCo

In [ ]:
import os

os.environ["MUJOCO_GL"] = "egl"

!nvidia-smi

## 2. Clone and install

In [ ]:
REPO_ROOT = "/content/wbc-mjlab"

if not os.path.isdir(REPO_ROOT):
  !git clone -q https://github.com/wbc-mjlab/wbc-mjlab.git {REPO_ROOT}

%cd {REPO_ROOT}

%pip install -q wbc-mjlab
print("✓ wbc-mjlab ready")

## 3. Convert bundled samples (one time)

Source CSVs are in the repo; NPZ clips are generated locally under `data/g1/samples/npz/`.

In [ ]:
!wbc-mjlab-data-to-npz --robot g1 --dataset samples --batch-size 16

## 4. Visualize samples

Scrub NPZ clips in Viser. Stop this cell (■) before the policy demo below.

In [ ]:
import subprocess
import sys

VISER_PORT = 8081

data_vis_process = subprocess.Popen(
  ["wbc-mjlab-data-vis", "--robot", "g1", "--dataset", "samples"],
  cwd=REPO_ROOT,
  stdout=subprocess.PIPE,
  stderr=subprocess.STDOUT,
  universal_newlines=True,
  bufsize=1,
)

for line in data_vis_process.stdout:
  print(line, end="")
  sys.stdout.flush()
  lower = line.lower()
  if any(s in lower for s in ("serving", "running on", "viser server", str(VISER_PORT))):
    print("\n✅ Motion viewer ready — run the next cell for the iframe.")
    break

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(VISER_PORT)

## 5. Launch policy demo

Runs bundled `demos/wbc_g1/model.pt` with 8 envs. Wait for the Viser message, then open the next cell.

In [ ]:
import subprocess
import sys

VISER_PORT = 8081

if "data_vis_process" in globals() and data_vis_process.poll() is None:
  data_vis_process.terminate()
  data_vis_process.wait(timeout=5)
  print("Stopped motion viewer.")

process = subprocess.Popen(
  ["wbc-mjlab-demo"],
  cwd=REPO_ROOT,
  stdout=subprocess.PIPE,
  stderr=subprocess.STDOUT,
  universal_newlines=True,
  bufsize=1,
)

for line in process.stdout:
  print(line, end="")
  sys.stdout.flush()
  lower = line.lower()
  if any(s in lower for s in ("serving", "running on", "viser server", str(VISER_PORT))):
    print("\n✅ Viser server is up — run the next cell for the iframe.")
    break

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(VISER_PORT)

## 6. Manual play (optional)

```bash
wbc-mjlab-play --task Wbc-G1 --dataset samples \
  --checkpoint-file demos/wbc_g1/model.pt --viewer viser --num-envs 8
```